In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
data = "fraud_detection_dataset_v2.csv"
df = pd.read_csv(data)
df.sample(5)

,transaction_id,timestamp,customer_id,merchant_id,transaction_type,amount,account_balance_before,account_balance_after,distance_from_home_km,new_payee,international_transaction,transactions_last_24h,failed_logins_last_week,merchant_category,device_trusted,ip_country,home_country,isFraud
4395,TXN-1004395,2026-01-04 00:53:38,CUST-11034,MERCH-8158,purchase,50.52,19565.84,19515.32,8.3,False,False,1,0,Dining,True,DE,DE,0
44425,TXN-1044425,2026-01-31 16:25:57,CUST-44251,MERCH-7968,purchase,254.91,1883.99,1629.08,129.7,False,False,2,0,Dining,True,DE,DE,0
38634,TXN-1038634,2026-01-27 15:01:55,CUST-96252,MERCH-2655,purchase,15.53,8867.21,8851.68,104.3,False,False,2,0,Luxury,True,UK,UK,0
29449,TXN-1029449,2026-01-21 07:01:40,CUST-61145,MERCH-8873,purchase,160.45,22399.75,22239.30,86.6,False,False,0,0,Entertainment,False,RU,RU,0
15575,TXN-1015575,2026-01-11 17:55:38,CUST-12126,MERCH-2367,purchase,233.90,1812.11,1578.21,185.2,False,False,7,0,Apparel,False,US,US,0


In [5]:
df.isnull().sum()

transaction_id               0
timestamp                    0
customer_id                  0
merchant_id                  0
transaction_type             0
amount                       0
account_balance_before       0
account_balance_after        0
distance_from_home_km        0
new_payee                    0
international_transaction    0
transactions_last_24h        0
failed_logins_last_week      0
merchant_category            0
device_trusted               0
ip_country                   0
home_country                 0
isFraud                      0
dtype: int64

In [6]:
df['isFraud'].value_counts()

isFraud
0    49251
1      749
Name: count, dtype: int64

In [7]:
df.groupby("international_transaction")["isFraud"].mean()

international_transaction
False    0.013408
True     0.029275
Name: isFraud, dtype: float64

In [8]:
df.groupby("new_payee")["isFraud"].mean()

new_payee
False    0.013202
True     0.028047
Name: isFraud, dtype: float64

In [9]:
df.groupby("device_trusted")["isFraud"].mean()

device_trusted
False    0.020735
True     0.013984
Name: isFraud, dtype: float64

In [10]:
df.groupby("international_transaction")["isFraud"].agg(["count","mean"])

,count,mean
international_transaction,,
False,45047,0.013408
True,4953,0.029275


In [11]:
pd.crosstab(
    [df["international_transaction"], df["new_payee"]],
    df["isFraud"],
    normalize="index"
)

isFraud                                     0         1
international_transaction new_payee                    
False                     False      0.986574  0.013426
                          True       0.986718  0.013282
True                      False      0.988823  0.011177
                          True       0.831283  0.168717

In [12]:
df["amount_balance_ratio"] = (
    df["amount"] / df["account_balance_before"]
)

In [13]:
df["country_mismatch"] = (
    df["ip_country"] != df["home_country"]
).astype(int)

In [14]:
df.groupby('country_mismatch')['isFraud'].mean()

country_mismatch
0    0.013408
1    0.029275
Name: isFraud, dtype: float64

In [15]:
df["high_velocity"] = (
    df["transactions_last_24h"] > 7
).astype(int)

In [16]:
df.groupby('high_velocity')['isFraud'].mean()

high_velocity
0    0.015077
1    0.014593
Name: isFraud, dtype: float64

In [17]:
df.groupby("high_velocity")["isFraud"].agg(["count","mean"])

,count,mean
high_velocity,,
0,39995,0.015077
1,10005,0.014593


In [18]:
(df["country_mismatch"] == df["international_transaction"]).mean()

1.0

In [19]:
df["weekend_transaction"] = (
    pd.to_datetime(df["timestamp"])
      .dt.dayofweek
      .isin([5,6])
).astype(int)

In [20]:
df["hour"] = pd.to_datetime(df["timestamp"]).dt.hour

df["night_transaction"] = (
    df["hour"].isin([0,1,2,3,4])
).astype(int)

In [21]:
df['velocity_amount_risk'] = df['transactions_last_24h'] * df['amount']

In [22]:
features = [
    "amount",
    "account_balance_before",
    "account_balance_after",
    "distance_from_home_km",
    "device_trusted",
    "new_payee",
    "international_transaction",
    "transactions_last_24h",
    "failed_logins_last_week",
    "merchant_category",
    "transaction_type",
    "velocity_amount_risk",
    "amount_balance_ratio"
]

In [23]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)


In [24]:
X = df[features]
y = df["isFraud"]

In [25]:
X = pd.get_dummies(
    X,
    columns=[
        "merchant_category",
        "transaction_type",
    ],
    drop_first=True
)

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)


RandomForestClassifier(class_weight='balanced', max_depth=10, n_estimators=200,
                       n_jobs=-1, random_state=42)

In [ ]:

probs = rf.predict_proba(X_test)[:, 1]

preds = (probs >= 0.40).astype(int)

: 

In [ ]:
print("ROC-AUC  :", roc_auc_score(y_test, probs))
print("Precision:", precision_score(y_test, preds))
print("Recall   :", recall_score(y_test, preds))
print("F1 Score :", f1_score(y_test, preds))

print("\nClassification Report")
print(classification_report(y_test, preds))

ROC-AUC  : 0.6689576988155668
Precision: 0.055323590814196244
Recall   : 0.35333333333333333
F1 Score : 0.09566787003610108

Classification Report
              precision    recall  f1-score   support

           0       0.99      0.91      0.95      9850
           1       0.06      0.35      0.10       150

    accuracy                           0.90     10000
   macro avg       0.52      0.63      0.52     10000
weighted avg       0.98      0.90      0.93     10000



: 

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Calculate class imbalance
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train)

probs = xgb.predict_proba(X_test)[:, 1]


preds = (probs >= 0.01).astype(int)

print("ROC-AUC  :", roc_auc_score(y_test, probs))
print("Precision:", precision_score(y_test, preds))
print("Recall   :", recall_score(y_test, preds))
print("F1 Score :", f1_score(y_test, preds))

print("\nClassification Report")
print(classification_report(y_test, preds))

: 

In [27]:
for t in [0.2, 0.25, 0.3, 0.35, 0.4]:
    preds = (probs >= t).astype(int)

    print(
        t,
        precision_score(y_test, preds),
        recall_score(y_test, preds),
        f1_score(y_test, preds)
    )

NameError: name 'probs' is not defined

In [28]:
for t in [0.45, 0.5, 0.55, 0.6]:
    preds = (probs >= t).astype(int)

    print(
        t,
        precision_score(y_test, preds),
        recall_score(y_test, preds),
        f1_score(y_test, preds)
    )

NameError: name 'probs' is not defined

In [29]:
importance = pd.Series(
    xgb.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(importance.head(15))

NameError: name 'xgb' is not defined

In [30]:
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    class_weight="balanced",
    verbose=-1
)

lgbm.fit(X_train, y_train)

probs = lgbm.predict_proba(X_test)[:, 1]

for t in [0.2, 0.25, 0.3, 0.35, 0.4]:
    preds = (probs >= t).astype(int)

    print(
        f"Threshold={t}",
        "Precision=", round(precision_score(y_test, preds), 3),
        "Recall=", round(recall_score(y_test, preds), 3),
        "F1=", round(f1_score(y_test, preds), 3)
    )

# Choose best threshold after inspecting results
preds = (probs >= 0.2).astype(int)

print("\nROC-AUC  :", roc_auc_score(y_test, probs))
print("Precision:", precision_score(y_test, preds))
print("Recall   :", recall_score(y_test, preds))
print("F1 Score :", f1_score(y_test, preds))

print("\nClassification Report")
print(classification_report(y_test, preds))

Threshold=0.2 Precision= 0.029 Recall= 0.62 F1= 0.056
Threshold=0.25 Precision= 0.038 Recall= 0.56 F1= 0.07
Threshold=0.3 Precision= 0.049 Recall= 0.507 F1= 0.09
Threshold=0.35 Precision= 0.067 Recall= 0.46 F1= 0.117
Threshold=0.4 Precision= 0.086 Recall= 0.393 F1= 0.141

ROC-AUC  : 0.7158443316412859
Precision: 0.029346797096875987
Recall   : 0.62
F1 Score : 0.056040976197649896

Classification Report
              precision    recall  f1-score   support

           0       0.99      0.69      0.81      9850
           1       0.03      0.62      0.06       150

    accuracy                           0.69     10000
   macro avg       0.51      0.65      0.43     10000
weighted avg       0.98      0.69      0.80     10000



In [34]:
from catboost import CatBoostClassifier
from sklearn.metrics import *

scale_weight = 40

cat = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    scale_pos_weight=scale_weight,  # <-- THIS IS CRITICAL TO FIX THE RECALL
    random_seed=42,
    verbose=100
)

cat.fit(X_train, y_train)

probs = cat.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, probs))
preds = (probs >= 0.4).astype(int)

print("PR-AUC       :", average_precision_score(y_test, probs))
print("Precision    :", precision_score(y_test, preds))
print("Recall       :", recall_score(y_test, preds))
print("F1 Score     :", f1_score(y_test, preds))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, preds))

0:	total: 70.1ms	remaining: 1m 10s
100:	total: 450ms	remaining: 4s
200:	total: 839ms	remaining: 3.33s
300:	total: 1.24s	remaining: 2.87s
400:	total: 1.62s	remaining: 2.42s
500:	total: 1.99s	remaining: 1.98s
600:	total: 2.36s	remaining: 1.57s
700:	total: 2.74s	remaining: 1.17s
800:	total: 3.11s	remaining: 773ms
900:	total: 3.49s	remaining: 383ms
999:	total: 3.85s	remaining: 0us
ROC-AUC: 0.7008561759729272
PR-AUC       : 0.22529331644352876
Precision    : 0.20535714285714285
Recall       : 0.30666666666666664
F1 Score     : 0.24598930481283424

Confusion Matrix
[[9672  178]
 [ 104   46]]


In [36]:
# 1. Get the raw importance scores
importances = cat.get_feature_importance(type="PredictionValuesChange")
feature_names = X_train.columns

# 2. Package it into a clean Pandas DataFrame
feature_imp_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print(feature_imp_df)

                            Feature  Importance
3             distance_from_home_km   15.734193
0                            amount   12.401217
9              velocity_amount_risk   11.817739
2             account_balance_after    9.086092
10             amount_balance_ratio    8.937538
1            account_balance_before    7.879329
7             transactions_last_24h    6.490123
8           failed_logins_last_week    4.440553
18        transaction_type_transfer    3.595952
6         international_transaction    3.195858
5                         new_payee    3.190097
4                    device_trusted    2.652430
19      transaction_type_withdrawal    1.845787
12         merchant_category_Dining    1.675370
15      merchant_category_Financial    1.354215
13    merchant_category_Electronics    1.330600
16        merchant_category_Grocery    1.205523
11        merchant_category_Apparel    1.155027
17         merchant_category_Luxury    1.053090
14  merchant_category_Entertainment    0

In [35]:
for t in [0.05,0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4,0.5,0.6,0.7,0.8]:
    preds = (probs >= t).astype(int)

    print(
        t,
        precision_score(y_test, preds),
        recall_score(y_test, preds),
        f1_score(y_test, preds),
        roc_auc_score(y_test, probs),
        average_precision_score(y_test, probs)
    )

0.05 0.02084528590552687 0.7266666666666667 0.040527979178285926 0.7008561759729272 0.22529331644352876
0.1 0.028455284552845527 0.56 0.05415860735009671 0.7008561759729272 0.22529331644352876
0.15 0.04044321329639889 0.4866666666666667 0.07468030690537085 0.7008561759729272 0.22529331644352876
0.2 0.05714285714285714 0.4266666666666667 0.10078740157480315 0.7008561759729272 0.22529331644352876
0.25 0.07810320781032078 0.37333333333333335 0.12918108419838523 0.7008561759729272 0.22529331644352876
0.3 0.1111111111111111 0.3466666666666667 0.16828478964401294 0.7008561759729272 0.22529331644352876
0.35 0.15457413249211358 0.32666666666666666 0.20985010706638116 0.7008561759729272 0.22529331644352876
0.4 0.20535714285714285 0.30666666666666664 0.24598930481283424 0.7008561759729272 0.22529331644352876
0.5 0.31297709923664124 0.2733333333333333 0.2918149466192171 0.7008561759729272 0.22529331644352876
0.6 0.449438202247191 0.26666666666666666 0.33472803347280333 0.7008561759729272 0.225293

In [32]:
frauds = X_test.copy()
frauds["actual"] = y_test
frauds["prob"] = probs

missed = frauds[
    (frauds["actual"] == 1) &
    (frauds["prob"] < 0.2)
]

print(missed.head(20))

       amount  account_balance_before  account_balance_after  \
30664  138.64                18089.87               17951.23   
24876  122.24                14194.89               14072.65   
36802  107.14                12660.38               12553.24   
21174   98.71                 6684.71                6586.00   
28652  133.49                  271.46                 137.97   
43148   28.35                 7750.14                7721.79   
25184    4.47                21676.38               21671.91   
32574  156.24                 6384.36                6228.12   
17976  119.84                 1716.69                1596.85   
47003   40.99                21916.60               21875.61   
34899    8.30                 1738.47                1730.17   
14455  535.99                 2389.95                1853.96   
26284   35.40                 1514.13                1478.73   
3958   325.72                11192.58               10866.86   
28159  216.88                16131.19   

In [89]:
# Save using native CatBoost method instead of joblib
cat.save_model("financial_fraud_model.cbm")

In [83]:
df.groupby("transaction_type")['isFraud'].mean()

transaction_type
purchase      0.012887
transfer      0.023109
withdrawal    0.013381
Name: isFraud, dtype: float64

In [84]:
# Find out what the maximum score the model gives to normal transactions is
normal_scores = cat.predict_proba(X_test[y_test == 0])[:, 1]
fraud_scores = cat.predict_proba(X_test[y_test == 1])[:, 1]

print(f"95th percentile of normal transactions: {np.percentile(normal_scores, 95)}")
print(f"Median score for actual fraud: {np.median(fraud_scores)}")

95th percentile of normal transactions: 0.280822236122494
Median score for actual fraud: 0.146846954970517


In [85]:
from sklearn.metrics import roc_auc_score, classification_report
import pandas as pd

# 1. Run predictions on your test set
test_probs = cat.predict_proba(X_test)[:, 1]

# 2. Check the AUC-ROC Score
auc = roc_auc_score(y_test, test_probs)
print(f"CRITICAL CHECK - AUC-ROC Score: {auc}")

# 3. Print full distribution descriptions
print("\n--- Clean Transactions Distribution ---")
print(pd.Series(cat.predict_proba(X_test[y_test == 0])[:, 1]).describe())

print("\n--- Fraud Transactions Distribution ---")
print(pd.Series(cat.predict_proba(X_test[y_test == 1])[:, 1]).describe())

CRITICAL CHECK - AUC-ROC Score: 0.7008561759729272

--- Clean Transactions Distribution ---
count    9850.000000
mean        0.086562
std         0.102000
min         0.000021
25%         0.020759
50%         0.052936
75%         0.114324
max         0.995734
dtype: float64

--- Fraud Transactions Distribution ---
count    150.000000
mean       0.321031
std        0.349228
min        0.000058
25%        0.047616
50%        0.146847
75%        0.662627
max        0.991200
dtype: float64


In [86]:
df.groupby('weekend_transaction')['isFraud'].mean()

weekend_transaction
0    0.015841
1    0.012902
Name: isFraud, dtype: float64